# MedGemma 27B Text — Persian medical evaluation

Evaluates `google/medgemma-27b-text-it`, served by **vLLM** behind an
**OpenAI-compatible API**, on two Persian medical benchmarks:

| Dataset | Type | What it probes | Metric |
|---|---|---|---|
| [PersianMedQA](https://huggingface.co/datasets/MohammadJRanjbar/PersianMedQA) | 4-way multiple choice, Iranian residency board exams | medical knowledge + reasoning in Persian | design-weighted accuracy (+ 95% CI with finite-population correction, macro-F1, parse-failure rate, choice bias, per-specialty) |
| [PerMedCQA](https://huggingface.co/datasets/NaghmehAI/PerMedCQA) | open-ended consumer questions with a doctor's reply | patient-language understanding + answer quality | LLM-as-judge 6-dimension rubric + reference-free/reference-based automatic metrics |

**Model under test:** `google/medgemma-27b-text-it` — the text-only 27B checkpoint
(Gemma-3 27B with medical post-training), served by vLLM on RunPod.

**Judge:** OpenAI `gpt-6-sol` over the official API — deliberately *not* a MedGemma
model, because a model grading its own family shows self-preference bias. It is a
reasoning model, so the notebook sends it `max_completion_tokens` and nothing else;
`temperature`/`top_p`/`seed` go only to the model under test, which is where
reproducible decoding actually matters. The open-ended scores are only as
trustworthy as the judge's independence.

**Pipeline:** load → stratified sample → prompt → concurrent API calls → parse →
metrics → LLM-judge → qualitative review → artifacts in `results/` → report.

**Reproducibility:** one `SEED` drives sampling *and* is passed as the vLLM request
`seed`, with `temperature=0`, so the model under test decodes deterministically. The
judge does not take those parameters and is the one stochastic element left. Every
configuration knob is in the *Configuration* cell, and every prediction is written to
CSV/JSON so any number here can be re-derived.

**Runtime:** the defaults are stratified samples of 1,000 PersianMedQA items and 500
PerMedCQA items (design in section 4) — 1,500 generations on the vLLM endpoint plus
500 reasoning-model judge calls to OpenAI, which are billed as tokens and are the
slowest and costliest part. Budget roughly 30–60 min at `CONCURRENCY=8`, less at
32–64. `EVAL_N_MCQ=0` turns PersianMedQA into a census (~5.2k generations, a couple
of GPU-hours); `EVAL_N_MCQ=440 EVAL_N_OPEN=10` is a short smoke run.

## 0. Requirements

`openai`, `datasets`, `pandas`, `tqdm` — all in this repo's `notebook` dependency
group, so `uv sync --group notebook` is the whole setup. Everything else — metrics,
statistics, HTTP — is the standard library.

In [1]:
import ast
import concurrent.futures as cf
import json
import math
import os
import platform
import random
import re
import time
import unicodedata
import urllib.request
from collections import Counter
from datetime import UTC, datetime
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import HTML, Markdown, display
from openai import OpenAI
from tqdm.auto import tqdm

RUN_STARTED = datetime.now(UTC)
TIMINGS: dict[str, float] = {}  # phase -> wall seconds, reported at the end

## 1. Configuration

Every knob the evaluation depends on. Values are read from the environment first
(so a CI run or a different pod needs no edit to this notebook), then from the
repo's `.env`, then from the defaults below.

In [2]:
def load_dotenv(path: Path) -> None:
    """Fill os.environ from a KEY=VALUE file, never overriding a real env var."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


load_dotenv(Path("../.env"))

# --- model under test --------------------------------------------------------
# One vLLM server speaking the OpenAI-compatible API. CHATBOT_* is this repo's
# existing MedGemma endpoint and is the fallback, so the notebook also runs
# unmodified against the service .env. The dict is keyed by a short display name;
# adding a second entry evaluates a second endpoint with no other change.
MODELS: dict[str, dict] = {
    "medgemma-27b-text": {
        "base_url": os.getenv("MEDGEMMA_TEXT_BASE_URL")
        or os.getenv("CHATBOT_BASE_URL", "http://localhost:8000/v1"),
        "model": os.getenv("MEDGEMMA_TEXT_MODEL")
        or os.getenv("CHATBOT_MODEL", "google/medgemma-27b-text-it"),
        "api_key": os.getenv("MEDGEMMA_TEXT_API_KEY") or os.getenv("CHATBOT_API_KEY", "not-needed"),
    },
}

# --- judge for the open-ended dataset ---------------------------------------
# OpenAI's official API, i.e. INDEPENDENT of the model under test: a model grading
# its own family shows self-preference bias. A reasoning model is the right choice
# for rubric grading, and `request_params()` below sends it the parameters it
# accepts (reasoning models reject temperature/top_p and want max_completion_tokens).
JUDGE: dict = {
    "base_url": os.getenv("JUDGE_BASE_URL", "https://api.openai.com/v1"),
    "model": os.getenv("JUDGE_MODEL", "gpt-6-sol"),
    "api_key": os.getenv("JUDGE_API_KEY", "not-needed"),
}
JUDGE_IS_INDEPENDENT = "medgemma" not in JUDGE["model"].lower()

# --- inference settings ------------------------------------------------------
SEED = int(os.getenv("EVAL_SEED", 42))  # sampling AND the vLLM request seed
TEMPERATURE = float(os.getenv("EVAL_TEMPERATURE", 0.0))  # greedy: an eval, not a demo
TOP_P = float(os.getenv("EVAL_TOP_P", 1.0))
MAX_TOKENS_MCQ = int(os.getenv("EVAL_MAX_TOKENS_MCQ", 512))  # short CoT + answer line
MAX_TOKENS_OPEN = int(os.getenv("EVAL_MAX_TOKENS_OPEN", 768))  # consumer-facing answer
# generous, because on a reasoning judge this budget covers the hidden reasoning
# tokens too — too small and the JSON verdict is truncated away entirely. A `-pro`
# model thinks longer than most, hence the headroom over the ~150 tokens of JSON.
MAX_TOKENS_JUDGE = int(os.getenv("EVAL_MAX_TOKENS_JUDGE", 8192))
CONCURRENCY = int(os.getenv("EVAL_CONCURRENCY", 8))  # in-flight requests per endpoint
REQUEST_TIMEOUT = float(os.getenv("EVAL_REQUEST_TIMEOUT", 300))
MAX_RETRIES = int(os.getenv("EVAL_MAX_RETRIES", 3))  # handled inside the OpenAI client

# --- evaluation size ---------------------------------------------------------
# PersianMedQA is sampled — ~5.2k test items is the expensive half of the run, and
# 1000 items estimate accuracy to about +-3pp, finer than any difference worth
# acting on. The design is in section 4. EVAL_N_MCQ=0 makes it a census.
N_PERSIANMEDQA = int(os.getenv("EVAL_N_MCQ", 1000))
MIN_PER_SPECIALTY = int(os.getenv("EVAL_MIN_PER_STRATUM", 20))  # floor per specialty
# PerMedCQA's train split is 64,279 items, so it is sampled too — a census would be
# ~64k generations plus ~64k reasoning-judge calls, which is absurd for a quality
# read. 500 items put the judge's mean scores inside about +-0.1 on a 1-5 scale.
N_PERMEDCQA = int(os.getenv("EVAL_N_OPEN", 500))
N_QUALITATIVE = 3  # examples of each kind printed in the qualitative section
N_RANDOM_SHOWCASE = 10  # unfiltered random items printed with the model's reply

# --- data --------------------------------------------------------------------
# Both are loaded with `datasets.load_dataset(repo, split=...)`, which handles the
# gate, the download and the on-disk cache. HF_TOKEN is picked up from the
# environment by huggingface_hub; PersianMedQA is gated and needs it.
HF_TOKEN = os.getenv("HF_TOKEN", "")
PERSIANMEDQA_REPO = "MohammadJRanjbar/PersianMedQA"
PERSIANMEDQA_SPLIT = os.getenv("PERSIANMEDQA_SPLIT", "test")  # train / validation / test
PERSIANMEDQA_LOCAL = os.getenv("PERSIANMEDQA_CSV", "")  # manual download fallback for the gate
PERMEDCQA_REPO = "NaghmehAI/PerMedCQA"
PERMEDCQA_SPLIT = os.getenv("PERMEDCQA_SPLIT", "train")  # train (64,279) / test (3,857)

RESULTS_DIR = Path(os.getenv("EVAL_RESULTS_DIR", "results/medgemma_persian_eval"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Free-text note for the report: the API cannot see the server's GPU.
HARDWARE_NOTE = os.getenv(
    "EVAL_HARDWARE_NOTE",
    "Serving hardware not reported by the API — fill EVAL_HARDWARE_NOTE with the "
    "actual GPU(s). MedGemma-27B in bfloat16 needs ~54 GB of weights, i.e. one "
    "80 GB H100/A100 (or two 40 GB cards with tensor parallelism).",
)


def redacted(cfg: dict) -> dict:
    """A config safe to print or save: never let an API key reach a file."""
    return cfg | {"api_key": "<set>" if cfg.get("api_key") else "<empty>"}


print(
    json.dumps(
        {
            "models": {name: redacted(cfg) for name, cfg in MODELS.items()},
            "judge": redacted(JUDGE),
            "seed": SEED,
            "temperature": TEMPERATURE,
            "n_mcq": N_PERSIANMEDQA,
            "n_open": N_PERMEDCQA,
            "concurrency": CONCURRENCY,
        },
        indent=2,
    )
)

{
  "models": {
    "medgemma-27b-text": {
      "base_url": "https://qc0cjvkf6po9k2-8000.proxy.runpod.net/v1",
      "model": "google/medgemma-27b-text-it",
      "api_key": "<set>"
    }
  },
  "judge": {
    "base_url": "https://api.gapgpt.app/v1",
    "model": "gpt-6-sol",
    "api_key": "<set>"
  },
  "seed": 42,
  "temperature": 0.0,
  "n_mcq": 1000,
  "n_open": 500,
  "concurrency": 8
}


## 2. Endpoint probe — vLLM serving configuration

`GET /v1/models` is the only serving detail the OpenAI-compatible surface exposes
(model id, context length, LoRA adapters), and it is read through the OpenAI SDK —
the RunPod proxy 403s a `Python-urllib` User-Agent and would make a healthy pod look
dead. vLLM's `/version` is queried too when the deployment exposes it. The probe also
checks that the endpoint actually advertises the model id you configured, which is
the difference between a working run and 404 on every request.

Whatever comes back is recorded verbatim in the report; anything the API cannot
report (GPU type, `--tensor-parallel-size`, dtype, `--gpu-memory-utilization`) has to
come from `HARDWARE_NOTE`.

In [3]:
def probe_endpoint(cfg: dict) -> dict:
    """Read back whatever the server is willing to say about itself.

    The model list goes through the OpenAI SDK rather than urllib on purpose: the
    RunPod proxy answers 403 to a `Python-urllib` User-Agent while serving the very
    same endpoint happily to the SDK, which made a healthy pod look dead here.
    """
    base = cfg["base_url"].rstrip("/")
    root = base[: -len("/v1")] if base.endswith("/v1") else base
    info: dict = {"base_url": cfg["base_url"], "configured_model": cfg["model"]}

    try:
        client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"], timeout=30, max_retries=1)
        served = [m.model_dump() for m in client.models.list().data]
    except Exception as exc:  # pod down, wrong key, wrong URL: record and move on
        return info | {"models": {"error": repr(exc)[:200]}, "reachable": False}

    # vLLM exposes its version outside /v1; most other providers do not
    request = urllib.request.Request(
        f"{root}/version",
        headers={"Authorization": f"Bearer {cfg['api_key']}", "User-Agent": "medgemma-eval"},
    )
    try:
        with urllib.request.urlopen(request, timeout=15) as response:
            info["version"] = json.loads(response.read())
    except Exception as exc:
        info["version"] = {"error": repr(exc)[:200]}

    entry = next((m for m in served if m.get("id") == cfg["model"]), None)
    detail = entry or (served[0] if served else {})
    return info | {
        "serves_configured_model": entry is not None,
        "max_model_len": detail.get("max_model_len"),
        # OpenAI advertises >100 models; only the head is worth recording
        "served_ids": [m.get("id") for m in served[:10]],
        "reachable": True,
    }


SERVING = {name: probe_endpoint(cfg) for name, cfg in MODELS.items()}
SERVING["judge"] = probe_endpoint(JUDGE)
for name, info in SERVING.items():
    flag = "OK " if info["reachable"] else "DOWN"
    served = info.get("served_ids", info.get("models"))
    print(f"[{flag}] {name}: {info['base_url']} -> {served}")
    if info["reachable"] and not info.get("serves_configured_model"):
        print(f"     !! this endpoint does not advertise {info['configured_model']!r}")

unreachable = [n for n, i in SERVING.items() if not i["reachable"]]
if unreachable:
    print(
        f"\n!! Not reachable: {unreachable}. Start the pods (or fix the *_BASE_URL) before running."
    )

[OK ] medgemma-27b-text: https://qc0cjvkf6po9k2-8000.proxy.runpod.net/v1 -> ['google/medgemma-27b-text-it']
[OK ] judge: https://api.gapgpt.app/v1 -> ['grok-4.6', 'deepseek-reasoner', 'gpt-5.3-codex', 'claude-3-haiku-20240307', 'text-embedding-v1', 'claude-opus-4-8', 'gapgpt-qwen-3.5-thinking', 'o3-mini-high', 'o4-mini', 'gpt-image-1-mini']


## 3. Datasets

### PersianMedQA
20,785 expert-validated 4-option questions from 14 years (2011–2024) of Iranian
national residency / pre-residency board exams, 23 specialties
([arXiv:2506.00250](https://arxiv.org/abs/2506.00250)). Loaded with
`load_dataset("MohammadJRanjbar/PersianMedQA", split="test")` — `PERSIANMEDQA_SPLIT`
picks among `train`/`validation`/`test`; the **test** split is the default and
train/val are left untouched.

The repo is **gated**: accept the terms on the dataset page while signed in, then
export `HF_TOKEN` (`load_dataset` picks it up from the environment). If that is not
possible, download the CSV by hand and point `PERSIANMEDQA_CSV` at it.

The rows do not store the options in four columns: `question` is the bare stem and
`answer` holds a Python dict literal `{'1': …, '2': …, '3': …, '4': …}`, with
`correct answer` an integer 1–4 indexing it, `field` the specialty and `year` the
Persian-calendar exam year. `PERSIANMEDQA_FIELDS` maps those names, `parse_options()`
unpacks the literal, and the loader fails loudly if a column disappears. **11 of the
5,236 test rows are dropped** — a blank option, three options instead of four, or a
stray extra key — leaving **5,225** usable four-way questions across 23 specialties.
The gold answers are near-uniform across A–D (24.3–26.5%), so always guessing one
letter scores ~26.5%: the floor any real accuracy has to beat.

### PerMedCQA
68,138 real consumer questions from four Iranian tele-health platforms (DrYab,
HiSalamat, GetZoop, Mavara-e-Teb, Nov 2022 – Apr 2024), each with the answering
doctor's reply ([arXiv:2505.18331](https://arxiv.org/abs/2505.18331)). Loaded with
`load_dataset("NaghmehAI/PerMedCQA", split="train")` — **64,279 items**, the split
`PERMEDCQA_SPLIT` selects (`test` holds the other 3,857). Ungated, so no token is
needed. Nothing is trained on here; "train" is simply the large split, and it is
sampled rather than run whole.

In [4]:
# --- PersianMedQA ------------------------------------------------------------
# The real shape of a row, read off the test split rather than guessed:
#   question_id     int  stable id
#   question        str  the stem — the options are NOT part of it
#   answer          str  a Python dict literal: "{'1': …, '2': …, '3': …, '4': …}"
#   correct answer  int  1..4, indexing that dict
#   field           str  specialty in Persian — the stratum
#   year            int  Persian-calendar exam year, 1390..1403
# English mirrors (question_english, …) and the question_type_* metadata are ignored:
# this evaluation is about Persian.
PERSIANMEDQA_FIELDS = {
    "id": "question_id",
    "question": "question",
    "options": "answer",
    "gold": "correct answer",
    "specialty": "field",
}


LETTERS = ["A", "B", "C", "D"]
_CHOICE_MAP = {
    **{letter: letter for letter in LETTERS},
    **{letter.lower(): letter for letter in LETTERS},
    **{str(i + 1): letter for i, letter in enumerate(LETTERS)},  # 1..4
    **{"۱۲۳۴"[i]: letter for i, letter in enumerate(LETTERS)},  # Persian digits
    **{"الف": "A", "ب": "B", "ج": "C", "د": "D"},  # Persian letters
}


def to_letter(value, options: list[str]) -> str | None:
    """Normalise a gold/predicted answer to A..D, matching option text if needed."""
    token = str(value).strip().strip(".)-(:").strip()
    if token in _CHOICE_MAP:
        return _CHOICE_MAP[token]
    for letter, text in zip(LETTERS, options, strict=True):
        if token and token == str(text).strip():
            return letter
    return None


def parse_options(raw) -> list[str] | None:
    """The four options out of the dict-literal string; None if the row is unusable.

    A handful of rows in the test split carry a blank option, only three options, or
    a stray extra key — they are dropped rather than scored, because a question with
    a missing distractor is not a four-way question.
    """
    try:
        parsed = ast.literal_eval(raw) if isinstance(raw, str) else raw
    except ValueError, SyntaxError:
        return None
    if not isinstance(parsed, dict):
        return None
    options = [str(parsed.get(str(i), "")).strip() for i in range(1, 5)]
    return options if all(options) else None


def load_persianmedqa() -> pd.DataFrame:
    raw = (
        pd.read_csv(PERSIANMEDQA_LOCAL)
        if PERSIANMEDQA_LOCAL
        else load_dataset(PERSIANMEDQA_REPO, split=PERSIANMEDQA_SPLIT).to_pandas()
    )
    absent = [c for c in PERSIANMEDQA_FIELDS.values() if c not in raw.columns]
    if absent:
        raise RuntimeError(
            f"PersianMedQA is missing {absent}; its columns are {list(raw.columns)}. "
            "Update PERSIANMEDQA_FIELDS if the dataset changed shape."
        )
    rows, dropped = [], 0
    for record in raw.to_dict("records"):
        options = parse_options(record[PERSIANMEDQA_FIELDS["options"]])
        if options is None:
            dropped += 1
            continue
        gold = to_letter(record[PERSIANMEDQA_FIELDS["gold"]], options)
        if gold is None:
            dropped += 1
            continue
        rows.append(
            {
                "id": f"pmq-{record[PERSIANMEDQA_FIELDS['id']]}",
                "question": str(record[PERSIANMEDQA_FIELDS["question"]]).strip(),
                **{f"opt_{letter}": text for letter, text in zip(LETTERS, options, strict=True)},
                "gold": gold,
                "specialty": str(record[PERSIANMEDQA_FIELDS["specialty"]]).strip() or "unknown",
            }
        )
    if dropped:
        print(f"PersianMedQA: dropped {dropped} rows with a missing option or unreadable key")
    return pd.DataFrame(rows)


# --- PerMedCQA ---------------------------------------------------------------
def load_permedcqa() -> pd.DataFrame:
    raw = load_dataset(PERMEDCQA_REPO, split=PERMEDCQA_SPLIT)
    df = pd.DataFrame(
        [
            {
                "id": f"pmc-{r['instance_id']}",
                "title": r.get("Title", ""),
                "question": r["Question"].strip(),
                "expert_answer": r["Expert_Answer"].strip(),
                "category": (r.get("Category") or "unknown").strip(),
                "specialty": (r.get("Specialty") or "unknown").strip(),
                "age": r.get("Age", ""),
                "sex": r.get("Sex", ""),
                "source": r.get("dataset_source", ""),
                "question_type": (r.get("QuestionTypeTag") or {}).get(
                    "QuestionType_Tag", "unknown"
                ),
            }
            for r in raw
        ]
    )
    return df[df.question.str.len() > 0].reset_index(drop=True)


t0 = time.perf_counter()
permedcqa = load_permedcqa()
try:
    persianmedqa = load_persianmedqa()
except Exception as exc:  # gated repo without a token is the usual cause
    print(f"PersianMedQA unavailable: {exc}")
    persianmedqa = pd.DataFrame()
TIMINGS["load_datasets"] = time.perf_counter() - t0

print(f"PersianMedQA: {len(persianmedqa)} usable items ({PERSIANMEDQA_SPLIT} split)")
display(persianmedqa.head(2))
print(f"PerMedCQA:    {len(permedcqa)} items ({PERMEDCQA_SPLIT} split)")

display(permedcqa.head(2))

PersianMedQA: dropped 11 rows with a missing option or unreadable key
PersianMedQA: 5225 usable items (test split)


,id,question,opt_A,opt_B,opt_C,opt_D,gold,specialty
0,pmq-1,در بیماری کواشیورکور، علائم زیر دیده می‌شود بجز:,تغییرات در پیگمان‌های پوستی و مو,ادم,کاهش وزن,کبد چرب,C,کودکان
1,pmq-2,در نوزاد پره‌ترم با دیسترس تنفسی، کدام گزینه د...,کدورت‌های Ground-glass,Air-bronchogram,ادم ریوی پلورال,Hypo-Aeration (کاهش هوادهی),C,رادیولوژی


PerMedCQA:    64279 items (train split)


,id,title,question,expert_answer,category,specialty,age,sex,source,question_type
0,pmc-5432,,با سلام و وقت بخیرخدمتتون من تو اقدام بارداری ...,سلام حداکثر دو روز حدود ۲۴ تا ۴۸ ساعت بعدش تخم...,زنان و زایمان,unknown,,,HiSalamat,Time(Other)
1,pmc-5179,بارداری؟,عنوان: بارداری؟\nسلام خانم دکتر،من هفته نهم با...,اگر محل تزريق هست نگران نباشيد,زنان و زايمان و نازايي,جراح و متخصص زنان و زايمان و نازايي,40,woman,DrYab,Side Effects


## 4. Sampling

**Both datasets are sampled**, because both splits are far larger than a sensible
run: PersianMedQA `test` is ~5.2k items (20,785 total − 14,549 train − 1,000 val) and
PerMedCQA `train` is 64,279. Every PersianMedQA item costs one generation; every
PerMedCQA item costs a generation *and* a reasoning-judge call, which is the
expensive one. `EVAL_N_MCQ=0` / `EVAL_N_OPEN=0` turns either into a census.

The two use the same sampler with deliberately different settings:

| | PersianMedQA | PerMedCQA |
|---|---|---|
| Stratum | specialty (23) | question type (18) |
| Default `n` | 1,000 | 500 |
| Floor per stratum | 20 | **none** |
| Headline statistic | design-weighted accuracy | unweighted mean judge score |

**Why no floor on PerMedCQA.** A floor over-samples small strata, which is only
harmless when the estimator weights them back down. PersianMedQA's accuracy does
exactly that (`design_weight`), but the judge's six dimensions are reported as plain
averages — so a floor there would quietly tilt the scores toward whatever the rare
question types (`Pronunciation`, `Overdose`, `Availability`) happen to elicit. Pure
proportional allocation keeps the sample's question-type mix equal to the split's,
which is what makes the unweighted mean unbiased.

### The design

**Stratified random sampling without replacement, specialty as the stratum, with a
floor and design weights.** Three decisions, each for a reason:

**1. Why stratify at all?** Accuracy differs sharply across the 23 specialties. Under
a simple random draw the *mix* of specialties is itself random, so part of the
measured accuracy is just which specialties happened to come up. Fixing the mix to
the population's removes that component of variance: a proportionally allocated
stratified sample is never less precise than an SRS of the same size, and here it is
meaningfully better. It also guarantees no specialty is missed entirely.

**2. Why a floor of `MIN_PER_SPECIALTY = 20`?** Pure proportional allocation gives a
specialty holding 1% of the split only ~10 items at n=1000 — too few to say anything
about it. Each stratum therefore gets 20 items first (or its whole population, if
smaller), and the remaining slots are allocated proportionally to the headroom left,
with largest-remainder rounding so the total is exactly `n`.

**3. Why design weights?** The floor deliberately over-samples small specialties, so
a plain average over sampled items is **biased** — it over-weights whatever the small
specialties happen to be good or bad at. Every sampled row therefore carries
`design_weight = N_h / n_h`, and the headline number is the stratified estimator

```
p̂  = Σ_h W_h · p̂_h                                    W_h = N_h / N
Var = Σ_h W_h² · (1 − n_h/N_h) · p̂_h(1 − p̂_h)/(n_h − 1)
```

The `(1 − n_h/N_h)` term is the **finite-population correction**: the frame is a fixed
~5.2k-item exam bank, not an infinite stream, so the interval narrows as coverage
grows and collapses to **exactly zero at a census**. That is why a Wilson (or any
binomial) interval is the wrong tool here — it assumes an SRS from an infinite
population, and would still report an interval after every item had been evaluated.
The unweighted sample accuracy is reported alongside, so the effect of the weights is
visible rather than hidden.

**Randomisation.** Within each stratum the rows are permuted with
`random.Random(f"{SEED}:{stratum}")` and the first `n_h` taken — seeded *per stratum
by name*, so one specialty's draw does not shift when another's allocation changes,
and the whole draw reproduces from `SEED` alone.

### What n=1000 buys

At p≈0.5 (the worst case) with the FPC, the 95% half-width is about ±2.8pp, and
narrower as accuracy moves away from 50%. Per specialty it is coarser — ~20–90 items
each, so ±10–20pp — enough to rank specialties and spot a collapse, not enough to
quote a per-specialty number. The realised allocation is printed below, so the design
is auditable rather than asserted.

In [5]:
def stratified_sample(
    df: pd.DataFrame, column: str, n: int, seed: int, min_per_stratum: int = 0
) -> pd.DataFrame:
    """Stratified sample without replacement, carrying its own design weights.

    Every stratum gets `min_per_stratum` rows first (or all of them, if smaller), then
    the remaining slots are allocated proportionally to the headroom left, with
    largest-remainder rounding. That floor over-samples small strata on purpose, so
    each row carries ``design_weight = N_h / n_h`` and ``stratified_accuracy`` weights
    by it — a plain mean over the sample would be biased toward the small strata.

    ``n <= 0`` or ``n >= len(df)`` is a census: every row, every weight 1.
    """
    keys = df[column].fillna("unknown")
    groups = {key: sorted(idx) for key, idx in df.groupby(keys).groups.items()}
    population = {key: len(members) for key, members in groups.items()}

    if n <= 0 or n >= len(df):
        take = dict(population)
    else:
        take = {key: min(min_per_stratum, size) for key, size in population.items()}
        remaining = n - sum(take.values())
        if remaining < 0:
            raise ValueError(
                f"min_per_stratum={min_per_stratum} across {len(groups)} strata needs "
                f"{sum(take.values())} items, more than n={n}"
            )
        headroom = {key: population[key] - take[key] for key in groups}
        spare = sum(headroom.values())
        quota = {key: remaining * headroom[key] / spare if spare else 0.0 for key in groups}
        extra = {key: min(int(value), headroom[key]) for key, value in quota.items()}
        order = sorted(groups, key=lambda k: (-(quota[k] - extra[k]), str(k)))
        leftover = remaining - sum(extra.values())
        while leftover > 0:  # largest remainder first, skipping exhausted strata
            before = leftover
            for key in order:
                if leftover and extra[key] < headroom[key]:
                    extra[key] += 1
                    leftover -= 1
            if leftover == before:
                break  # every stratum exhausted: the sample is the whole frame
        take = {key: take[key] + extra[key] for key in groups}

    picked = []
    for key, members in groups.items():
        # seeded per stratum BY NAME: one stratum's draw never shifts because another
        # stratum's allocation changed, and the whole draw reproduces from `seed`
        picked.extend(random.Random(f"{seed}:{key}").sample(members, len(members))[: take[key]])

    sample = df.loc[sorted(picked)].copy()
    drawn = sample[column].fillna("unknown")
    sample["stratum_population"] = drawn.map(population)
    sample["stratum_sampled"] = drawn.map(take)
    sample["design_weight"] = sample.stratum_population / sample.stratum_sampled
    return sample.reset_index(drop=True)


mcq_sample = (
    stratified_sample(persianmedqa, "specialty", N_PERSIANMEDQA, SEED, MIN_PER_SPECIALTY)
    if len(persianmedqa)
    else persianmedqa
)
open_sample = stratified_sample(permedcqa, "question_type", N_PERMEDCQA, SEED)

for label, sample, population in (
    ("PersianMedQA", mcq_sample, persianmedqa),
    ("PerMedCQA", open_sample, permedcqa),
):
    mode = "FULL SPLIT (census)" if len(sample) == len(population) else "stratified sample"
    print(f"{label:13} {len(sample)} / {len(population)} ({mode})")

if len(mcq_sample):  # the realised allocation, so the design is auditable
    display(
        mcq_sample.groupby("specialty")
        .agg(
            population=("stratum_population", "first"),
            sampled=("stratum_sampled", "first"),
            design_weight=("design_weight", "first"),
        )
        .sort_values("population", ascending=False)
        .round(2)
    )

PersianMedQA  1000 / 5225 (stratified sample)
PerMedCQA     500 / 64279 (stratified sample)


,population,sampled,design_weight
specialty,,,
کودکان,629,89,7.07
جراحی,618,88,7.02
زنان,440,68,6.47
عفونی,270,48,5.62
پاتولوژی,243,45,5.40
نورولوژی,219,42,5.21
ارتوپدی,214,42,5.10
روانپزشکی,190,39,4.87
غدد,184,39,4.72


## 5. Prompts

One prompt per dataset, fixed for the whole run and recorded verbatim in
`run_config.json` — a prompt edited mid-run makes every number before and after it
incomparable, and is the easiest way to accidentally benchmark the prompt instead of
the model.

* **PersianMedQA.** Persian system + user message, options labelled `A`–`D`, a short
  chain-of-thought (≤ 3 sentences, which the PersianMedQA paper found helps open
  models) and a mandated final line `پاسخ: X`. The fixed final line is what makes
  extraction deterministic instead of a guessing game; the parser still falls back
  to looser patterns so a format slip is counted as a *wrong answer only if it
  really is one*, and format failures are reported separately.
* **PerMedCQA.** Persian system prompt casting the model as a clinician answering a
  patient on a tele-health platform, with the patient's age/sex given as context
  exactly as the platform would. Answer capped at ~150 words to match the register
  of the reference replies (median expert answer is ~93 characters — terse), and
  asked for plain Persian rather than textbook prose.

Neither prompt is few-shot: the point is zero-shot behaviour of the served model,
which is how it is actually used in this repo.

In [6]:
MCQ_SYSTEM = (
    "شما یک پزشک متخصص هستید که به سؤالات چندگزینه‌ای آزمون‌های دستیاری و پیش‌کارورزی "
    "پزشکی ایران پاسخ می‌دهید. همیشه دقیقاً یکی از گزینه‌ها را انتخاب می‌کنید."
)

MCQ_TEMPLATE = """سؤال زیر یک سؤال چهارگزینه‌ای پزشکی است. صحیح‌ترین گزینه را انتخاب کن.

سؤال: {question}

A) {opt_A}
B) {opt_B}
C) {opt_C}
D) {opt_D}

ابتدا حداکثر در سه جمله استدلال کوتاه خود را بنویس.
سپس در خط آخر، دقیقاً به این شکل و بدون هیچ توضیح اضافه، پاسخ نهایی را بنویس:
پاسخ: X
که در آن X یکی از حروف A، B، C یا D است."""

OPEN_SYSTEM = (
    "شما یک پزشک فارسی‌زبان هستید که در یک سامانه پرسش و پاسخ آنلاین به سؤالات پزشکی "
    "مراجعان پاسخ می‌دهید. پاسخ‌ها باید دقیق، مبتنی بر شواهد، قابل فهم برای فرد غیرمتخصص "
    "و به زبان فارسی روان باشند."
)

OPEN_TEMPLATE = """اطلاعات مراجع: سن {age} سال، جنسیت {sex}.

پرسش مراجع:
{question}

به عنوان پزشک پاسخ بده:
- فقط به زبان فارسی و حداکثر در ۱۵۰ کلمه بنویس.
- مستقیماً به همان چیزی که مراجع پرسیده پاسخ بده.
- در صورت لزوم به داروها، دوز یا اقدام تشخیصی مشخص اشاره کن.
- اگر وضعیت نیازمند مراجعه حضوری یا اورژانسی است، آن را صریح بگو."""

_SEX_FA = {"man": "مرد", "woman": "زن", "male": "مرد", "female": "زن"}


def mcq_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": MCQ_SYSTEM},
        {
            "role": "user",
            "content": MCQ_TEMPLATE.format(
                question=row.question,
                opt_A=row.opt_A,
                opt_B=row.opt_B,
                opt_C=row.opt_C,
                opt_D=row.opt_D,
            ),
        },
    ]


def open_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": OPEN_SYSTEM},
        {
            "role": "user",
            "content": OPEN_TEMPLATE.format(
                age=row.age or "نامشخص",
                sex=_SEX_FA.get(str(row.sex).lower(), "نامشخص"),
                question=row.question,
            ),
        },
    ]


print(
    mcq_messages(mcq_sample.iloc[0])[1]["content"]
    if len(mcq_sample)
    else "(PersianMedQA unavailable)"
)
print("\n" + "=" * 70 + "\n")
print(open_messages(open_sample.iloc[0])[1]["content"])

سؤال زیر یک سؤال چهارگزینه‌ای پزشکی است. صحیح‌ترین گزینه را انتخاب کن.

سؤال: فرد 20 ساله‌ای که پدر وی دچار اختلال اسکیزوفرنی می‌باشد و به دنبال بحران‌های اجتماعی سال گذشته مهاجرت کرده است. همراه برادر زندگی می‌کند و از چند ماه قبل علائم سایکوز آشکار شده است. رابطه با برادر پرتنش و دارای تعارض می‌باشد. کدامیک از موارد زیر، Predisposing factors می‌باشد؟

A) اختلال اسکیزوفرنی پدر
B) مهاجرت
C) روابط پرتنش با برادر
D) بحران‌های اجتماعی

ابتدا حداکثر در سه جمله استدلال کوتاه خود را بنویس.
سپس در خط آخر، دقیقاً به این شکل و بدون هیچ توضیح اضافه، پاسخ نهایی را بنویس:
پاسخ: X
که در آن X یکی از حروف A، B، C یا D است.


اطلاعات مراجع: سن 26 سال، جنسیت زن.

پرسش مراجع:
عنوان: فشارخون بارداری؟
سلام بنده 26 سالمه 12 هفته باردارم از چندیدن سال پیش در محیط های ناآشنا و مطب ها دچار لرق دست وپا ولرزش دست میشدم همزمان فشار خونم هم بالا ببود 16 یا 17 و ضرباک قلبم هم 115 تا میزد نوار قلبش مشکلی نداشت الان بد ونود حاملگیم نگرانم که مشکلی برای بچم پیش بیاد آیا نن فسار خون گرفتم وبازد دارو مصرف کنم ای علامته

## 6. Answer extraction, text metrics, statistics

**Extraction.** The mandated `پاسخ: X` line is looked for last-match-first (so a
letter mentioned mid-reasoning does not win), accepting `پاسخ`/`جواب`/`answer`,
Latin letters, `1`–`4`, Persian digits `۱`–`۴` and `الف/ب/ج/د`. If that fails, a
bare `A)`-style option marker on the final line is accepted. If that fails too the
prediction is `None` — a **format failure**, reported separately from a wrong
answer, because the two mean different things.

**Automatic text metrics** for PerMedCQA (all reference-based ones are weak signals
against a terse doctor reply, so they support the judge rather than replace it):

* `persian_ratio` — share of letters in the Arabic/Persian block: catches
  language drift, the single most common failure of non-Persian-centric models.
* `latin_ratio` — English leakage.
* `lcs_f1` — overlap F1 against the expert answer over normalised tokens.
* `length_ratio` — answer length ÷ expert length: verbosity against the reference register.

**Statistics.** The design-based estimator of section 4: accuracy is the
population-weighted `Σ W_h·p̂_h`, and its 95% interval comes from the stratified
variance with the finite-population correction, so a census reports no interval.

**Request parameters.** `request_params()` lives here rather than with the runner so
the self-check below can reach it: vLLM and classic chat models are sent
`temperature`/`top_p`/`max_tokens`/`seed`, while a reasoning model (`gpt-5*`,
`gpt-6*`, `o1/o3/o4*`) is sent `max_completion_tokens` alone — it rejects the others
outright.

In [7]:
_ANSWER_RE = re.compile(
    r"(?:پاسخ|جواب|answer)\s*(?:نهایی|final)?\s*[:：\-–]?\s*\(?\s*([A-Da-d1-4۱-۴]|الف|ب|ج|د)\s*\)?",
    re.IGNORECASE,
)
_BARE_RE = re.compile(r"(?:^|\s)\(?([A-D])\)?(?:[\.\):\s]|$)")


def extract_choice(text: str) -> str | None:
    """Pull the selected option out of a model reply; None means format failure."""
    if not text:
        return None
    matches = _ANSWER_RE.findall(text)
    if matches:
        return _CHOICE_MAP.get(matches[-1].strip())
    tail = text.strip().splitlines()[-1] if text.strip() else ""
    bare = _BARE_RE.findall(tail) or _BARE_RE.findall(text)
    return bare[-1] if bare else None


_FA_FIX = str.maketrans({"ي": "ی", "ك": "ک", "‌": " ", "ۀ": "ه", "أ": "ا", "إ": "ا", "آ": "ا"})


def normalize_fa(text: str) -> str:
    return unicodedata.normalize("NFKC", text or "").translate(_FA_FIX)


def tokens(text: str) -> list[str]:
    return re.findall(r"\w+", normalize_fa(text).lower())


def persian_ratio(text: str) -> float:
    letters = [c for c in (text or "") if c.isalpha()]
    if not letters:
        return 0.0
    return round(sum("؀" <= c <= "ۿ" for c in letters) / len(letters), 3)


def latin_ratio(text: str) -> float:
    letters = [c for c in (text or "") if c.isalpha()]
    if not letters:
        return 0.0
    return round(sum("a" <= c.lower() <= "z" for c in letters) / len(letters), 3)


def lcs_f1(reference: str, hypothesis: str) -> float:
    """Token-overlap F1 against the reference answer.

    ponytail: difflib's contiguous matching blocks, not a true LCS DP — it under-counts
    reordered overlap by a few points. Swap in rouge-score if the exact ROUGE-L number
    ever has to line up with a published table.
    """
    ref, hyp = tokens(reference), tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    matched = sum(
        b.size for b in SequenceMatcher(None, ref, hyp, autojunk=False).get_matching_blocks()
    )
    if not matched:
        return 0.0
    precision, recall = matched / len(hyp), matched / len(ref)
    return round(2 * precision * recall / (precision + recall), 4)


def stratified_accuracy(frame: pd.DataFrame, stratum: str = "specialty") -> dict:
    """Population accuracy under the stratified design, with a 95% interval.

    p_hat = sum_h W_h * p_h with W_h = N_h/N, and the textbook stratified variance
    with the finite-population correction (1 - n_h/N_h). A census therefore gets an
    interval of width zero, which is right: nothing was left unsampled. `n_h` counts
    the items that actually returned a reply, so API errors shrink their stratum
    rather than silently scoring as wrong answers.
    """
    total = frame.groupby(stratum).stratum_population.first().sum()
    estimate = variance = 0.0
    for _, group in frame.groupby(stratum):
        pop, drawn = float(group.stratum_population.iloc[0]), len(group)
        share, rate = pop / total, float(group.correct.mean())
        estimate += share * rate
        if drawn > 1:
            variance += share**2 * (1 - drawn / pop) * rate * (1 - rate) / (drawn - 1)
    half = 1.96 * math.sqrt(variance)
    return {
        "accuracy": round(estimate, 4),
        "ci_low": round(max(0.0, estimate - half), 4),
        "ci_high": round(min(1.0, estimate + half), 4),
        "se": round(math.sqrt(variance), 4),
    }


def macro_f1(golds: list, predictions: list, labels=tuple(LETTERS)) -> float:
    """Macro-F1 over the four options — exposes a model that always answers 'C'."""
    scores = []
    for label in labels:
        tp = sum(g == label and p == label for g, p in zip(golds, predictions, strict=True))
        fp = sum(g != label and p == label for g, p in zip(golds, predictions, strict=True))
        fn = sum(g == label and p != label for g, p in zip(golds, predictions, strict=True))
        scores.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return round(sum(scores) / len(scores), 4)


_REASONING_RE = re.compile(r"^(gpt-5|gpt-6|o[134])", re.IGNORECASE)


def request_params(model: str, max_tokens: int) -> dict:
    """Decoding parameters this endpoint will actually accept.

    Reasoning models (gpt-5/gpt-6 families, o-series) reject `temperature`, `top_p`
    and `max_tokens`, and count hidden reasoning tokens against
    `max_completion_tokens`. vLLM and the classic chat models take the full set, and
    greedy decoding there is what makes the model-under-test run reproducible.
    """
    if _REASONING_RE.match(model):
        return {"max_completion_tokens": max_tokens}
    return {
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": max_tokens,
        "seed": SEED,
    }

### Self-check

Runs in under a second and fails loudly if answer extraction, the design-based
estimator, the sampler or the per-endpoint request parameters break. Every number
downstream is built on those functions, so this is the cheapest place to catch a
regression.

In [8]:
def _self_check() -> None:
    assert extract_choice("استدلال ... \nپاسخ: B") == "B"
    assert extract_choice("گزینه A محتمل است اما ...\nپاسخ: C") == "C", (
        "must take the final answer line"
    )
    assert extract_choice("پاسخ: ۳") == "C", "Persian digits"
    assert extract_choice("جواب: الف") == "A", "Persian option letters"
    assert extract_choice("Answer: (d)") == "D"
    assert extract_choice("D") == "D", "bare letter fallback"
    assert extract_choice("نمی‌دانم") is None and extract_choice("") is None, (
        "format failure stays None"
    )

    assert to_letter(3, ["x", "y", "z", "w"]) == "C", "gold is a 1..4 index"
    assert to_letter("y", ["x", "y", "z", "w"]) == "B", "gold given as option text"

    assert parse_options("{'1': 'a ', '2': 'b', '3': 'c', '4': 'd'}") == ["a", "b", "c", "d"]
    assert parse_options("{'1': 'a', '2': 'b', '3': 'c', '4': 'd', '90': 'x'}") == list("abcd"), (
        "a stray extra key must not break the row"
    )
    assert parse_options("{'1': 'a', '2': '', '3': 'c', '4': 'd'}") is None, "blank option"
    assert parse_options("{'1': 'a', '2': 'b', '3': 'c'}") is None, "only three options"
    assert parse_options("not a dict") is None and parse_options(None) is None

    srs = pd.DataFrame(
        {
            "specialty": ["a"] * 100,
            "stratum_population": 10_000,
            "correct": [True] * 50 + [False] * 50,
        }
    )
    out = stratified_accuracy(srs)  # a single stratum reduces to SRS with an FPC
    assert out["accuracy"] == 0.5 and 0.097 < out["ci_high"] - out["accuracy"] < 0.099, out
    census = pd.DataFrame(
        {"specialty": ["a"] * 10, "stratum_population": 10, "correct": [True] * 5 + [False] * 5}
    )
    assert stratified_accuracy(census)["se"] == 0.0, "a census has no sampling error"
    skewed = pd.DataFrame(
        {
            "specialty": ["big"] * 10 + ["small"] * 10,
            "stratum_population": [900] * 10 + [100] * 10,
            "correct": [True] * 10 + [False] * 10,
        }
    )
    assert stratified_accuracy(skewed)["accuracy"] == 0.9, (
        "weights follow the population, not the sample (unweighted would say 0.5)"
    )

    vllm_params = request_params("google/medgemma-27b-text-it", 512)
    assert vllm_params["max_tokens"] == 512 and vllm_params["temperature"] == TEMPERATURE
    assert vllm_params["seed"] == SEED, "the model under test must decode reproducibly"
    judge_params = request_params("gpt-6-sol", 8192)
    assert judge_params == {"max_completion_tokens": 8192}, (
        "a reasoning judge rejects temperature/top_p/max_tokens"
    )

    assert macro_f1(["A", "B"], ["A", "B"]) == 0.5, "2 of 4 labels perfect, 2 absent"
    assert macro_f1(["A", "B", "C", "D"], ["C"] * 4) == 0.1, "always-C collapses macro-F1"

    assert persian_ratio("سلام دنیا") == 1.0 and persian_ratio("hello") == 0.0
    assert 0.4 < lcs_f1("سردرد و تب دارم", "سردرد و تب دارم") + 0 <= 1.0
    assert lcs_f1("الف", "") == 0.0

    frame = pd.DataFrame({"s": ["a"] * 60 + ["b"] * 30 + ["c"] * 10, "v": range(100)})
    plain = stratified_sample(frame, "s", 10, SEED)
    assert len(plain) == 10 and plain.s.value_counts().to_dict() == {"a": 6, "b": 3, "c": 1}, (
        "proportional allocation when no floor is asked for"
    )
    floored = stratified_sample(frame, "s", 20, SEED, min_per_stratum=5)
    counts = floored.s.value_counts().to_dict()
    assert len(floored) == 20 and min(counts.values()) >= 5, counts
    assert counts["a"] > counts["c"], "the floor must not flatten the allocation"
    assert round(floored.design_weight.sum()) == 100, (
        "design weights must sum back to the population"
    )
    try:  # a floor that cannot fit in n must say so, not silently overshoot
        stratified_sample(frame, "s", 10, SEED, min_per_stratum=5)
        raise AssertionError("expected ValueError: 3 strata x 5 > n=10")
    except ValueError:
        pass
    everything = stratified_sample(frame, "s", 0, SEED)
    assert len(everything) == 100 and (everything.design_weight == 1).all(), "n<=0 is a census"
    assert plain.v.tolist() == stratified_sample(frame, "s", 10, SEED).v.tolist(), (
        "seed must be stable"
    )
    assert plain.v.tolist() != stratified_sample(frame, "s", 10, SEED + 1).v.tolist(), (
        "seed must matter"
    )
    print("self-check passed")


_self_check()

self-check passed


## 7. Inference runner

One `OpenAI` client per endpoint, `CONCURRENCY` requests in flight (8 by default,
with a `tqdm` bar per phase), retries handled by the SDK (`max_retries`), and `seed`
forwarded to vLLM so a re-run of the same sample reproduces the same generations.

`request_params()` (section 6) decides what each endpoint is actually sent, and a
reasoning judge's hidden reasoning tokens come out of the same
`max_completion_tokens` budget, which is why `MAX_TOKENS_JUDGE` is generous.

Every call records latency, token usage and
`finish_reason` (`length` = the answer was truncated, which the qualitative section
counts as its own error class). A failed call is recorded as a row with an `error`
rather than aborting the run — a dead pod mid-sweep should not cost the whole sweep.

In [9]:
def make_client(cfg: dict) -> OpenAI:
    return OpenAI(
        base_url=cfg["base_url"],
        api_key=cfg["api_key"],
        timeout=REQUEST_TIMEOUT,
        max_retries=MAX_RETRIES,
    )


def chat(client: OpenAI, model: str, messages: list[dict], max_tokens: int) -> dict:
    started = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            **request_params(model, max_tokens),
        )
        choice = response.choices[0]
        usage = response.usage
        return {
            "text": (choice.message.content or "").strip(),
            "finish_reason": choice.finish_reason,
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "latency_s": round(time.perf_counter() - started, 2),
            "error": None,
        }
    except Exception as exc:
        return {
            "text": "",
            "finish_reason": "error",
            "prompt_tokens": None,
            "completion_tokens": None,
            "latency_s": round(time.perf_counter() - started, 2),
            "error": repr(exc)[:300],
        }


def run_batch(cfg: dict, conversations: list[list[dict]], max_tokens: int, desc: str) -> list[dict]:
    """Fan `conversations` out over one endpoint, preserving input order."""
    client = make_client(cfg)
    results: list[dict | None] = [None] * len(conversations)
    with cf.ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
        futures = {
            pool.submit(chat, client, cfg["model"], messages, max_tokens): i
            for i, messages in enumerate(conversations)
        }
        for future in tqdm(cf.as_completed(futures), total=len(futures), desc=desc):
            results[futures[future]] = future.result()
    return results  # type: ignore[return-value]

## 8. PersianMedQA — run and score

In [10]:
mcq_frames = []
if len(mcq_sample):
    conversations = [mcq_messages(row) for row in mcq_sample.itertuples()]
    for name, cfg in MODELS.items():
        started = time.perf_counter()
        replies = run_batch(cfg, conversations, MAX_TOKENS_MCQ, f"PersianMedQA · {name}")
        TIMINGS[f"persianmedqa::{name}"] = time.perf_counter() - started
        frame = mcq_sample.copy()
        frame["model"] = name
        frame["served_model"] = cfg["model"]
        for field in (
            "text",
            "finish_reason",
            "prompt_tokens",
            "completion_tokens",
            "latency_s",
            "error",
        ):
            frame[field] = [r[field] for r in replies]
        frame["prediction"] = frame.text.map(extract_choice)
        frame["correct"] = frame.prediction == frame.gold
        frame["format_failure"] = frame.prediction.isna() & frame.error.isna()
        mcq_frames.append(frame)
    mcq_results = pd.concat(mcq_frames, ignore_index=True)
    mcq_results.to_csv(RESULTS_DIR / "persianmedqa_predictions.csv", index=False)
    print(f"wrote {len(mcq_results)} rows to {RESULTS_DIR / 'persianmedqa_predictions.csv'}")
else:
    mcq_results = pd.DataFrame()
    print("PersianMedQA skipped — dataset not loaded (see section 3).")

PersianMedQA · medgemma-27b-text:   0%|          | 0/1000 [00:00<?, ?it/s]

wrote 1000 rows to results/medgemma_persian_eval/persianmedqa_predictions.csv


In [11]:
def mcq_summary(frame: pd.DataFrame) -> dict:
    answered = frame[frame.error.isna()]
    estimate = (
        stratified_accuracy(answered)
        if len(answered)
        else {"accuracy": 0.0, "ci_low": 0.0, "ci_high": 0.0, "se": 0.0}
    )
    return {
        "model": frame.model.iloc[0],
        "served_model": frame.served_model.iloc[0],
        "n": len(frame),
        "api_errors": int(frame.error.notna().sum()),
        # design-weighted estimate for the whole split, then the raw sample mean so
        # the effect of the weights stays visible
        "accuracy": estimate["accuracy"],
        "ci_low": estimate["ci_low"],
        "ci_high": estimate["ci_high"],
        "se": estimate["se"],
        "accuracy_unweighted": round(float(answered.correct.mean()), 4) if len(answered) else 0.0,
        "macro_f1": macro_f1(answered.gold.tolist(), answered.prediction.fillna("-").tolist()),
        "format_failure_rate": round(frame.format_failure.mean(), 4),
        "truncated_rate": round((frame.finish_reason == "length").mean(), 4),
        "mean_latency_s": round(frame.latency_s.mean(), 2),
        "mean_completion_tokens": round(frame.completion_tokens.dropna().mean(), 1)
        if frame.completion_tokens.notna().any()
        else None,
    }


MCQ_SUMMARY = pd.DataFrame([mcq_summary(f) for f in mcq_frames]) if mcq_frames else pd.DataFrame()

if len(MCQ_SUMMARY):
    display(MCQ_SUMMARY)

    # choice distribution — a model that always answers "C" scores 25% for the wrong reason
    display(
        pd.DataFrame(
            {
                f.model.iloc[0]: f.prediction.fillna("(none)").value_counts(normalize=True).round(3)
                for f in mcq_frames
            }
        )
        .fillna(0)
        .sort_index()
        .rename_axis("chosen option")
    )

    per_specialty = (
        pd.concat(mcq_frames)
        .assign(correct=lambda d: d.correct.astype(int))
        .pivot_table(
            index="specialty", columns="model", values="correct", aggfunc=["mean", "count"]
        )
    )
    display(per_specialty.round(3).head(25))

,model,served_model,n,api_errors,accuracy,ci_low,ci_high,se,accuracy_unweighted,macro_f1,format_failure_rate,truncated_rate,mean_latency_s,mean_completion_tokens
0,medgemma-27b-text,google/medgemma-27b-text-it,1000,0,0.6635,0.6367,0.6903,0.0137,0.665,0.6655,0.001,0.003,10.17,156.1


,medgemma-27b-text
chosen option,
(none),0.001
A,0.268
B,0.255
C,0.246
D,0.230


,mean,count
model,medgemma-27b-text,medgemma-27b-text
specialty,,
آمار,0.611,36
اخلاق,0.630,27
ارتوپدی,0.571,42
اورولوژی,0.605,38
جراحی,0.534,88
خون و انکولوژی,0.657,35
رادیولوژی,0.789,38
روانپزشکی,0.667,39


## 9. PerMedCQA — generation

In [12]:
open_conversations = [open_messages(row) for row in open_sample.itertuples()]
open_frames = []
for name, cfg in MODELS.items():
    started = time.perf_counter()
    replies = run_batch(cfg, open_conversations, MAX_TOKENS_OPEN, f"PerMedCQA · {name}")
    TIMINGS[f"permedcqa::{name}"] = time.perf_counter() - started
    frame = open_sample.copy()
    frame["model"] = name
    frame["served_model"] = cfg["model"]
    for field in (
        "text",
        "finish_reason",
        "prompt_tokens",
        "completion_tokens",
        "latency_s",
        "error",
    ):
        frame[field] = [r[field] for r in replies]
    open_frames.append(frame)

open_results = pd.concat(open_frames, ignore_index=True)
print(open_results.groupby("model")[["latency_s", "completion_tokens"]].mean().round(2))

PerMedCQA · medgemma-27b-text:   0%|          | 0/500 [00:00<?, ?it/s]

                   latency_s  completion_tokens
model                                          
medgemma-27b-text       16.0              234.6


## 10. PerMedCQA — evaluation methodology

A doctor's reply on a tele-health platform is one of many acceptable answers and is
often a single terse sentence, so n-gram overlap with it is close to meaningless as
a quality score. The evaluation therefore has two layers:

**(a) Automatic, reference-free / reference-based signals** — cheap, deterministic,
and they catch the failures that a judge is bad at spotting consistently:
`persian_ratio` (language drift), `latin_ratio` (English leakage),
`length_ratio` vs the expert reply (verbosity), `lcs_f1` (topical overlap),
truncation rate, API error rate.

**(b) Reference-guided LLM-as-a-judge**, 1–5 on six dimensions, exactly the axes
requested:

| Dimension | Question the judge answers |
|---|---|
| `medical_correctness` | Is the clinical content right and safe, given the doctor's reply as reference? |
| `relevance` | Does it answer *this* patient's actual question? |
| `completeness` | Are the key elements of the reference reply covered? |
| `persian_fluency` | Is it natural, grammatical, fully-Persian prose? |
| `terminology` | Is Persian medical terminology used correctly (drug names, anatomy, procedures)? |
| `patient_language_understanding` | Did it correctly read colloquial/vague patient phrasing, slang and typos? |

The judge grades **one answer at a time** (no pairwise ordering, so no position
bias), sees the expert reply as a reference but is told it is *one* acceptable
answer rather than the only one, must return JSON, and also tags each answer with
zero or more labels from a **fixed error taxonomy** — that taxonomy is what the
"common recurring errors" section is counted from, rather than impressions.

**Known limitations, stated up front.** The judge defaults to OpenAI `gpt-6-sol`,
which is independent of the model under test — point `JUDGE_MODEL` back at a MedGemma
checkpoint and the scores pick up self-preference bias, which the report then flags.
Independence is not calibration, though: a single LLM's 1–5 scale is anchored by
nothing but its own prior, so the absolute numbers mean less than the ranking across
dimensions and question types. A reasoning judge also spends part of
`MAX_TOKENS_JUDGE` on hidden reasoning, and a verdict cut off mid-JSON is counted as
a parse failure rather than a score — watch that count. A human spot-check of ~30
items against the same six dimensions is the recommended third layer before any
number here is published.

In [13]:
ERROR_TAXONOMY = [
    "hallucinated_fact",  # states something not true / not supported
    "contradicts_expert",  # directly conflicts with the reference reply
    "unsafe_advice",  # could harm if followed
    "missing_key_info",  # omits what the reference considered essential
    "off_topic",  # answers a different question
    "non_persian_or_mixed_language",  # drifts to English/Arabic, or code-switches
    "wrong_terminology",  # misuses a medical term or drug name
    "misread_patient_question",  # misunderstood colloquial phrasing
    "generic_deferral_only",  # only "see a doctor", no content
    "overlong_or_repetitive",
]

JUDGE_SYSTEM = (
    "You are a strict medical evaluation judge fluent in Persian (Farsi) and clinical "
    "medicine. You grade a model's Persian answer to a real patient question. Output "
    "JSON only — no prose, no markdown fence."
)

JUDGE_TEMPLATE = """Patient question (Persian):
\"\"\"{question}\"\"\"

Reference answer written by a licensed physician (Persian). It is ONE acceptable
answer, not the only one — do not penalise an answer merely for differing in style,
length or extra correct detail:
\"\"\"{expert}\"\"\"

Model answer to grade (Persian):
\"\"\"{candidate}\"\"\"

Score each dimension on an integer 1-5 scale (1 = unacceptable, 3 = adequate, 5 = excellent):
- medical_correctness: clinical accuracy and safety of the content.
- relevance: does it answer THIS patient's actual question?
- completeness: does it cover the key elements the reference covers?
- persian_fluency: natural, grammatical, fully Persian prose.
- terminology: correct Persian medical terminology, drug names, anatomy.
- patient_language_understanding: did it correctly interpret colloquial, vague or
  misspelled patient phrasing, including implicit context?

Also list zero or more error labels from EXACTLY this set (use [] if none):
{taxonomy}

Return only this JSON object:
{{"medical_correctness": int, "relevance": int, "completeness": int,
 "persian_fluency": int, "terminology": int, "patient_language_understanding": int,
 "errors": [string], "rationale": "one English sentence"}}"""

JUDGE_DIMENSIONS = [
    "medical_correctness",
    "relevance",
    "completeness",
    "persian_fluency",
    "terminology",
    "patient_language_understanding",
]


def judge_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": JUDGE_SYSTEM},
        {
            "role": "user",
            "content": JUDGE_TEMPLATE.format(
                question=row.question,
                expert=row.expert_answer,
                candidate=row.text or "(empty answer)",
                taxonomy=", ".join(ERROR_TAXONOMY),
            ),
        },
    ]


def parse_judgement(text: str) -> dict:
    """Pull the JSON verdict out of a judge reply; scores clamped to 1-5."""
    empty = {"judge_parse_ok": False, "errors": [], "rationale": ""} | dict.fromkeys(
        JUDGE_DIMENSIONS
    )
    match = re.search(r"\{.*\}", text or "", re.S)
    if not match:
        return empty
    try:
        raw = json.loads(match.group(0))
    except json.JSONDecodeError:
        return empty
    verdict: dict = {"judge_parse_ok": True}
    for dimension in JUDGE_DIMENSIONS:
        value = raw.get(dimension)
        verdict[dimension] = min(5, max(1, int(value))) if isinstance(value, (int, float)) else None
    verdict["errors"] = [e for e in raw.get("errors", []) if e in ERROR_TAXONOMY]
    verdict["rationale"] = str(raw.get("rationale", ""))[:400]
    return verdict

In [14]:
# automatic metrics first — they need no judge and are computed for every answer
open_results["persian_ratio"] = open_results.text.map(persian_ratio)
open_results["latin_ratio"] = open_results.text.map(latin_ratio)
open_results["lcs_f1"] = [lcs_f1(r.expert_answer, r.text) for r in open_results.itertuples()]
open_results["length_ratio"] = (
    open_results.text.str.len() / open_results.expert_answer.str.len().clip(lower=1)
).round(2)
open_results["truncated"] = open_results.finish_reason == "length"

display(
    open_results.groupby("model")[
        ["persian_ratio", "latin_ratio", "lcs_f1", "length_ratio", "truncated"]
    ]
    .mean()
    .round(3)
)

,persian_ratio,latin_ratio,lcs_f1,length_ratio,truncated
model,,,,,
medgemma-27b-text,0.992,0.008,0.067,10.422,0.0


In [15]:
started = time.perf_counter()
judgements = run_batch(
    JUDGE,
    [judge_messages(r) for r in open_results.itertuples()],
    MAX_TOKENS_JUDGE,
    f"judge · {JUDGE['model']}",
)
TIMINGS["permedcqa::judge"] = time.perf_counter() - started

verdicts = pd.DataFrame(
    [parse_judgement(j["text"]) | {"judge_error": j["error"]} for j in judgements]
)
open_results = pd.concat([open_results.reset_index(drop=True), verdicts], axis=1)
open_results["judge_mean"] = open_results[JUDGE_DIMENSIONS].mean(axis=1).round(3)
open_results.to_csv(RESULTS_DIR / "permedcqa_predictions.csv", index=False)

unparsed = int((~open_results.judge_parse_ok.fillna(False)).sum())
print(f"judge parse failures: {unparsed} / {len(open_results)}")
OPEN_SUMMARY = (
    open_results.groupby("model")
    .agg(
        n=("id", "size"),
        api_errors=("error", lambda s: int(s.notna().sum())),
        **{d: (d, "mean") for d in JUDGE_DIMENSIONS},
        judge_mean=("judge_mean", "mean"),
        persian_ratio=("persian_ratio", "mean"),
        lcs_f1=("lcs_f1", "mean"),
        length_ratio=("length_ratio", "mean"),
        truncated=("truncated", "mean"),
        mean_latency_s=("latency_s", "mean"),
    )
    .round(3)
)
display(OPEN_SUMMARY)

judge · gpt-6-sol:   0%|          | 0/500 [00:00<?, ?it/s]

judge parse failures: 7 / 500


,n,api_errors,medical_correctness,relevance,completeness,persian_fluency,terminology,patient_language_understanding,judge_mean,persian_ratio,lcs_f1,length_ratio,truncated,mean_latency_s
model,,,,,,,,,,,,,,
medgemma-27b-text,500,0,3.552,4.473,3.428,4.905,4.406,4.394,4.193,0.992,0.067,10.422,0.0,16.002


In [16]:
# recurring errors, straight off the judge's fixed taxonomy
ERROR_COUNTS = (
    pd.DataFrame(
        {
            name: Counter(e for row in frame.errors.dropna() for e in row)
            for name, frame in open_results.groupby("model")
        }
    )
    .reindex(ERROR_TAXONOMY)
    .fillna(0)
    .astype(int)
)
ERROR_COUNTS = ERROR_COUNTS.sort_values(by=ERROR_COUNTS.columns[0], ascending=False)
display(ERROR_COUNTS)

# per question-type breakdown: where does patient-style Persian actually break down?
display(
    open_results.pivot_table(
        index="question_type", columns="model", values="judge_mean", aggfunc="mean"
    ).round(2)
)

,medgemma-27b-text
missing_key_info,270
hallucinated_fact,116
contradicts_expert,64
wrong_terminology,58
misread_patient_question,48
unsafe_advice,42
generic_deferral_only,11
overlong_or_repetitive,6
non_persian_or_mixed_language,3
off_topic,1


model,medgemma-27b-text
question_type,
Action,4.50
Action/Time,2.67
Alternatives,4.21
Availability,3.50
Combination,4.43
Comparison,3.96
Contraindication,4.29
Dose,3.93
Indication,4.25


## 11. Qualitative analysis

Concrete outputs, rendered right-to-left: correct, incorrect and unparseable
multiple-choice answers, then the best and worst open-ended answers by judge score,
then an unfiltered random sample. These are the examples quoted in the report — read
them before trusting any aggregate, particularly the judge's.

In [17]:
RTL_STYLE = "text-align:right;border-right:3px solid #888;padding:6px 10px;margin:6px 0"


def rtl(title: str, body: str) -> None:
    display(
        HTML(
            f'<div dir="rtl" style="{RTL_STYLE}">'
            f'<b style="color:#666">{title}</b><br>'
            f"{body.replace(chr(10), '<br>')}</div>"
        )
    )


QUALITATIVE: dict[str, list[dict]] = {}
if len(mcq_results):
    for name, frame in mcq_results.groupby("model"):
        answered = frame[frame.error.isna()]
        picks = []
        for label, subset in (
            ("CORRECT", answered[answered.correct]),
            ("INCORRECT", answered[~answered.correct & answered.prediction.notna()]),
            ("FORMAT FAILURE", answered[answered.prediction.isna()]),
        ):
            for row in subset.head(N_QUALITATIVE).itertuples():
                picks.append(
                    {
                        "kind": label,
                        "id": row.id,
                        "specialty": row.specialty,
                        "question": row.question,
                        "gold": row.gold,
                        "prediction": row.prediction,
                        "answer": row.text,
                    }
                )
                rtl(
                    f"[{name}] {label} · {row.specialty} · gold={row.gold} pred={row.prediction}",
                    f"{row.question[:400]}<br><br><i>{(row.text or '')[:600]}</i>",
                )
        QUALITATIVE[f"persianmedqa::{name}"] = picks

In [18]:
for name, frame in open_results.groupby("model"):
    scored = frame[frame.judge_mean.notna()].sort_values("judge_mean")
    picks = []
    for label, subset in (
        ("WORST", scored.head(N_QUALITATIVE)),
        ("BEST", scored.tail(N_QUALITATIVE).iloc[::-1]),
    ):
        for row in subset.itertuples():
            picks.append(
                {
                    "kind": label,
                    "id": row.id,
                    "question_type": row.question_type,
                    "judge_mean": row.judge_mean,
                    "errors": row.errors,
                    "question": row.question,
                    "expert_answer": row.expert_answer,
                    "answer": row.text,
                    "rationale": row.rationale,
                }
            )
            rtl(
                f"[{name}] {label} · {row.question_type} · judge={row.judge_mean} · {row.errors}",
                f"<u>پرسش</u>: {row.question[:400]}<br><br>"
                f"<u>پاسخ پزشک</u>: {row.expert_answer[:300]}<br><br>"
                f"<u>پاسخ مدل</u>: <i>{(row.text or '')[:800]}</i>",
            )
            print(f"   judge rationale: {row.rationale}")
    QUALITATIVE[f"permedcqa::{name}"] = picks

   judge rationale: The answer mistakes Ramenoflor for ramipril and appears to interpret a concern about breathing after a dose as bad breath, so it misses the relevant guidance on administration, continuing treatment, and possible causes of drooling despite appropriate emergency advice for breathing difficulty.


   judge rationale: The answer misidentifies Venostat and incorrectly presents increased appetite as a known adverse effect, while omitting the physician’s key points about thyroid testing and taking the medication with fatty meals.


   judge rationale: The patient asks whether this clinician performs laser treatment for a pilonidal cyst, but the answer incorrectly says yes, equates it with an epidermoid cyst, and omits the referral to a surgeon.


   judge rationale: The answer correctly addresses the neighboring tooth’s progressive breakage, explains that an in-person examination is needed to determine treatment, and gives reasonable treatment possibilities.


   judge rationale: The answer correctly addresses management of PCOS and reassures the patient that pregnancy is often possible with appropriate treatment.


   judge rationale: The answer appropriately explains that optic nerve damage cannot be confirmed from the history alone and requires an in-person eye examination and review of the images.


### 10 random items with the model's reply

The best/worst picks above are selected by score, which flatters and damns on
purpose. This is the unfiltered version: `N_RANDOM_SHOWCASE` items drawn at random
from each dataset with the raw reply underneath, so the typical answer is visible
rather than only the extremes. Seeded, so a re-run shows the same items.

In [19]:
def showcase(results: pd.DataFrame, label: str, header, body) -> list:
    """Print N_RANDOM_SHOWCASE random items with the model's reply underneath."""
    if not len(results):
        return []
    ids = sorted(results.id.unique())
    chosen = random.Random(f"{SEED}:{label}").sample(ids, min(N_RANDOM_SHOWCASE, len(ids)))
    for item_id in chosen:
        rows = results[results.id == item_id]
        head = rows.iloc[0]
        rtl(*header(head))
        for row in rows.itertuples():
            rtl(*body(row))
    return chosen


mcq_shown = showcase(
    mcq_results,
    "showcase-mcq",
    lambda head: (
        f"{head.id} · {head.specialty} · gold = {head.gold}",
        head.question[:500]
        + "<br><br>"
        + "<br>".join(f"{letter}) {head[f'opt_{letter}']}" for letter in LETTERS),
    ),
    lambda row: (
        f"↳ {row.model} · pred = {row.prediction} {'✓' if row.correct else '✗'}",
        f"<i>{(row.text or row.error or '(empty)')[:700]}</i>",
    ),
)

open_shown = showcase(
    open_results,
    "showcase-open",
    lambda head: (
        f"{head.id} · {head.question_type} · {head.category} · سن {head.age}",
        f"<u>پرسش</u>: {head.question[:600]}<br><br><u>پاسخ پزشک</u>: {head.expert_answer[:400]}",
    ),
    lambda row: (
        f"↳ {row.model} · judge = {row.judge_mean} · fa = {row.persian_ratio} · {row.errors}",
        f"<i>{(row.text or row.error or '(empty)')[:900]}</i>",
    ),
)

QUALITATIVE["random_showcase"] = {"persianmedqa": mcq_shown, "permedcqa": open_shown}

## 12. Artifacts

Everything needed to re-derive or audit any number above, in `results/medgemma_persian_eval/`:

Written, never read back: every run queries the model and overwrites these.

| File | Contents |
|---|---|
| `persianmedqa_predictions.csv` | one row per (item, model): prompt inputs, raw reply, extracted choice, gold, correctness, latency, tokens |
| `permedcqa_predictions.csv` | one row per (item, model): raw reply, automatic metrics, six judge scores, error labels, judge rationale |
| `summary.json` | aggregate metrics, per-phase timings, error tallies |
| `run_config.json` | models, endpoints, serving probe, prompts, decoding params, seed, sample sizes, client host |
| `qualitative_examples.json` | the examples printed above, including the ids of the random sample |
| `report.md` | the final report |

Point `EVAL_RESULTS_DIR` at a new folder to keep a previous run's files.

In [20]:
RUN_CONFIG = {
    "run_started_utc": RUN_STARTED.isoformat(),
    "models": {name: redacted(cfg) for name, cfg in MODELS.items()},
    "judge": redacted(JUDGE) | {"independent_of_models_under_test": JUDGE_IS_INDEPENDENT},
    "serving_probe": SERVING,
    "inference": {
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed": SEED,
        "max_tokens_mcq": MAX_TOKENS_MCQ,
        "max_tokens_open": MAX_TOKENS_OPEN,
        "max_tokens_judge": MAX_TOKENS_JUDGE,
        "concurrency": CONCURRENCY,
        "request_timeout_s": REQUEST_TIMEOUT,
        "max_retries": MAX_RETRIES,
    },
    "datasets": {
        "persianmedqa": {
            "repo": PERSIANMEDQA_REPO,
            "split": PERSIANMEDQA_SPLIT,
            "population": len(persianmedqa),
            "sampled": len(mcq_sample),
            "stratified_by": "specialty",
        },
        "permedcqa": {
            "repo": PERMEDCQA_REPO,
            "split": PERMEDCQA_SPLIT,
            "population": len(permedcqa),
            "sampled": len(open_sample),
            "stratified_by": "question_type",
        },
    },
    "prompts": {
        "mcq_system": MCQ_SYSTEM,
        "mcq_template": MCQ_TEMPLATE,
        "open_system": OPEN_SYSTEM,
        "open_template": OPEN_TEMPLATE,
        "judge_system": JUDGE_SYSTEM,
        "judge_template": JUDGE_TEMPLATE,
    },
    "client_host": {"python": platform.python_version(), "platform": platform.platform()},
    "hardware_note": HARDWARE_NOTE,
}

SUMMARY = {
    "persianmedqa": MCQ_SUMMARY.to_dict("records") if len(MCQ_SUMMARY) else [],
    "permedcqa": OPEN_SUMMARY.reset_index().to_dict("records"),
    "permedcqa_errors": ERROR_COUNTS.to_dict(),
    "timings_s": {k: round(v, 1) for k, v in TIMINGS.items()},
    "total_wall_s": round((datetime.now(UTC) - RUN_STARTED).total_seconds(), 1),
}

(RESULTS_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2))
(RESULTS_DIR / "summary.json").write_text(
    json.dumps(SUMMARY, ensure_ascii=False, indent=2, default=str)
)
(RESULTS_DIR / "qualitative_examples.json").write_text(
    json.dumps(QUALITATIVE, ensure_ascii=False, indent=2, default=str)
)
print("\n".join(str(p) for p in sorted(RESULTS_DIR.iterdir())))

results/medgemma_persian_eval/permedcqa_predictions.csv
results/medgemma_persian_eval/persianmedqa_predictions.csv
results/medgemma_persian_eval/qualitative_examples.json
results/medgemma_persian_eval/run_config.json
results/medgemma_persian_eval/summary.json


## 13. Final report

Generated from the numbers computed above — nothing here is typed by hand, so
re-running the notebook regenerates a report that matches its own artifacts.

In [21]:
def fmt_cell(value) -> str:
    return f"{value:.4g}" if isinstance(value, float) else str(value)


def fmt_table(frame: pd.DataFrame) -> str:
    """Markdown table without pulling in tabulate for six lines of work."""
    if not len(frame):
        return "_no data_"
    header = [str(c) for c in frame.columns]
    return "\n".join(
        [
            "| " + " | ".join(header) + " |",
            "|" + "---|" * len(header),
            *(
                "| " + " | ".join(fmt_cell(v) for v in row) + " |"
                for row in frame.itertuples(index=False)
            ),
        ]
    )


def build_report() -> str:
    mcq_source = PERSIANMEDQA_LOCAL or f"{PERSIANMEDQA_REPO} [{PERSIANMEDQA_SPLIT}]"
    mcq_strata = "" if len(mcq_sample) == len(persianmedqa) else ", stratified by specialty"
    open_strata = "" if len(open_sample) == len(permedcqa) else ", stratified by question type"
    lines = [
        f"# MedGemma 27B on Persian medical benchmarks — {RUN_STARTED:%Y-%m-%d}",
        "",
        "## 1. Models and serving",
        "",
        "| Role | Served model id | Endpoint | Reachable | max_model_len |",
        "|---|---|---|---|---|",
    ]
    for name, cfg in MODELS.items():
        info = SERVING[name]
        lines.append(
            f"| {name} | `{cfg['model']}` | `{cfg['base_url']}` | "
            f"{info['reachable']} | {info.get('max_model_len', 'n/a')} |"
        )
    lines += [
        f"| judge | `{JUDGE['model']}` | `{JUDGE['base_url']}` | {SERVING['judge']['reachable']} | "
        f"{SERVING['judge'].get('max_model_len', 'n/a')} |",
        "",
        "Served by vLLM over its OpenAI-compatible API. Serving flags beyond what "
        "`GET /v1/models` reports are not visible to a client; see `run_config.json` "
        "→ `serving_probe` for the raw response.",
        "",
        f"**Hardware:** {HARDWARE_NOTE}",
        "",
        f"**Client:** Python {platform.python_version()} on {platform.platform()}, "
        f"`openai` SDK, {CONCURRENCY} concurrent requests per endpoint.",
        "",
        "## 2. Inference settings",
        "",
        f"Model under test: `temperature={TEMPERATURE}`, `top_p={TOP_P}`, `seed={SEED}` "
        f"(sent to vLLM), `max_tokens` {MAX_TOKENS_MCQ} (MCQ) / {MAX_TOKENS_OPEN} "
        "(open-ended). Greedy decoding: the run is an evaluation, not a demo, so "
        "variance is removed where it can be. Judge: "
        + (
            f"`max_completion_tokens={MAX_TOKENS_JUDGE}` only — a reasoning model rejects "
            "temperature/top_p and spends part of that budget on hidden reasoning tokens."
            if _REASONING_RE.match(JUDGE["model"])
            else f"the same decoding parameters, `max_tokens={MAX_TOKENS_JUDGE}`."
        )
        + f" `timeout={REQUEST_TIMEOUT}s`, `max_retries={MAX_RETRIES}`.",
        "",
        "## 3. Data and sampling",
        "",
        f"* **PersianMedQA** `{mcq_source}` — "
        f"{len(mcq_sample)} of {len(persianmedqa)} items{mcq_strata}."
        + (
            ""
            if len(mcq_sample)
            else "  **NOT RUN** — the dataset could not be loaded (gated repo: "
            "accept the terms and set `HF_TOKEN`, or set `PERSIANMEDQA_CSV`)."
        ),
        f"* **PerMedCQA** `{PERMEDCQA_REPO}` [{PERMEDCQA_SPLIT}] — "
        f"{len(open_sample)} of {len(permedcqa)} items{open_strata}.",
        "",
        (
            "PersianMedQA: full split, no sampling."
            if len(mcq_sample) == len(persianmedqa)
            else "PersianMedQA: stratified random sample without replacement, specialty as "
            f"stratum, a floor of {MIN_PER_SPECIALTY} items per specialty, the rest "
            "allocated proportionally with largest-remainder rounding; accuracy is the "
            "design-weighted estimator with a finite-population correction, so the interval "
            "reflects the sampling design rather than a binomial assumption. Seeded with "
            f"`SEED={SEED}`."
        )
        + (
            " PerMedCQA: full split, no sampling."
            if len(open_sample) == len(permedcqa)
            else f" PerMedCQA: stratified sample, seeded with `SEED={SEED}`."
        )
        + " Both models see the identical items, so the comparison is paired.",
        "",
        "## 4. Prompts",
        "",
        "Identical for both checkpoints; full text in `run_config.json` → `prompts`. "
        "PersianMedQA: Persian system prompt + options A–D + ≤3-sentence CoT + mandated "
        "final line `پاسخ: X`. PerMedCQA: Persian clinician system prompt, patient age/sex "
        "as context, ≤150-word Persian answer.",
        "",
        "## 5. PersianMedQA results",
        "",
        fmt_table(MCQ_SUMMARY),
        "",
        "Metrics: `accuracy` is the design-weighted estimate for the whole split "
        "(sum of W_h * p_h) with a 95% interval from the stratified variance including "
        "the finite-population correction; `accuracy_unweighted` is the raw sample mean, "
        "for comparison. Plus macro-F1 across the four options (catches option bias), "
        "format-failure rate (reply with no extractable choice) and truncation rate. "
        "Items whose API call failed are excluded and counted under `api_errors`.",
        "",
    ]
    lines += [
        "## 6. PerMedCQA results",
        "",
        fmt_table(OPEN_SUMMARY.reset_index()),
        "",
        f"Judge: `{JUDGE['model']}` — "
        + (
            "independent of the models under test."
            if JUDGE_IS_INDEPENDENT
            else "**the same family as the models under test, so the scores carry "
            "self-preference bias; treat them as indicative and re-run with an "
            "independent judge before publishing.**"
        ),
        "",
        "Six dimensions scored 1–5 (medical correctness, relevance, completeness, Persian "
        "fluency, terminology, patient-language understanding), reference-guided against "
        "the physician's reply, one answer at a time so there is no position bias. "
        "Automatic metrics alongside: Persian-character ratio (language drift), token-overlap "
        "F1 against the expert reply, length ratio, truncation rate.",
        "",
        "## 7. Persian understanding and medical terminology",
        "",
        "Persian-character ratio per model: "
        + ", ".join(f"`{m}` {v:.3f}" for m, v in OPEN_SUMMARY.persian_ratio.items())
        + "; terminology score: "
        + ", ".join(f"`{m}` {v:.2f}/5" for m, v in OPEN_SUMMARY.terminology.items())
        + "; patient-language understanding: "
        + ", ".join(
            f"`{m}` {v:.2f}/5" for m, v in OPEN_SUMMARY.patient_language_understanding.items()
        )
        + ".",
        "",
        "Per-question-type judge means (which patient intents break down) are in "
        "`summary.json`; the taxonomy tally below is the evidence for recurring errors.",
        "",
        "## 8. Recurring errors",
        "",
        fmt_table(ERROR_COUNTS.reset_index().rename(columns={"index": "error"})),
        "",
        "## 9. Examples",
        "",
        "Correct / incorrect MCQ answers, best / worst open-ended answers, and a random "
        "unfiltered sample of 10 items per dataset are in `qualitative_examples.json` and "
        "rendered in section 11 of the notebook.",
        "",
        "## 10. Cost and execution time",
        "",
        "| Phase | Wall seconds |",
        "|---|---|",
    ]
    lines += [f"| {phase} | {seconds:.1f} |" for phase, seconds in SUMMARY["timings_s"].items()]
    lines += [
        f"| **total** | **{SUMMARY['total_wall_s']:.1f}** |",
        "",
        f"At `CONCURRENCY={CONCURRENCY}`. Client-side cost is negligible; the constraint is "
        "GPU memory on the serving side — MedGemma-27B in bf16 is ~54 GB of weights, so one "
        "80 GB H100/A100 (or two 40 GB cards with tensor parallelism). The judge runs on "
        "OpenAI's API and costs tokens, not GPU time.",
        "",
        "## 11. Caveats",
        "",
        "* Single greedy sample per item — no self-consistency, no prompt-variation study.",
        "* The judge is a single LLM. Its scores are calibrated by nothing but its own "
        "prior; a ~30-item human review on the same six dimensions is the recommended "
        "confirmation step before quoting them.",
        "* Accuracy is computed over items that returned a reply; API errors are reported "
        "separately rather than silently scored as wrong.",
    ]
    return "\n".join(lines)


REPORT = build_report()
(RESULTS_DIR / "report.md").write_text(REPORT)
display(Markdown(REPORT))

# MedGemma 27B on Persian medical benchmarks — 2026-09-26

## 1. Models and serving

| Role | Served model id | Endpoint | Reachable | max_model_len |
|---|---|---|---|---|
| medgemma-27b-text | `google/medgemma-27b-text-it` | `https://qc0cjvkf6po9k2-8000.proxy.runpod.net/v1` | True | 8128 |
| judge | `gpt-6-sol` | `https://api.gapgpt.app/v1` | True | None |

Served by vLLM over its OpenAI-compatible API. Serving flags beyond what `GET /v1/models` reports are not visible to a client; see `run_config.json` → `serving_probe` for the raw response.

**Hardware:** Serving hardware not reported by the API — fill EVAL_HARDWARE_NOTE with the actual GPU(s). MedGemma-27B in bfloat16 needs ~54 GB of weights, i.e. one 80 GB H100/A100 (or two 40 GB cards with tensor parallelism).

**Client:** Python 3.14.0 on macOS-14.3-arm64-arm-64bit-Mach-O, `openai` SDK, 8 concurrent requests per endpoint.

## 2. Inference settings

Model under test: `temperature=0.0`, `top_p=1.0`, `seed=42` (sent to vLLM), `max_tokens` 512 (MCQ) / 768 (open-ended). Greedy decoding: the run is an evaluation, not a demo, so variance is removed where it can be. Judge: `max_completion_tokens=8192` only — a reasoning model rejects temperature/top_p and spends part of that budget on hidden reasoning tokens. `timeout=300.0s`, `max_retries=3`.

## 3. Data and sampling

* **PersianMedQA** `MohammadJRanjbar/PersianMedQA [test]` — 1000 of 5225 items, stratified by specialty.
* **PerMedCQA** `NaghmehAI/PerMedCQA` [train] — 500 of 64279 items, stratified by question type.

PersianMedQA: stratified random sample without replacement, specialty as stratum, a floor of 20 items per specialty, the rest allocated proportionally with largest-remainder rounding; accuracy is the design-weighted estimator with a finite-population correction, so the interval reflects the sampling design rather than a binomial assumption. Seeded with `SEED=42`. PerMedCQA: stratified sample, seeded with `SEED=42`. Both models see the identical items, so the comparison is paired.

## 4. Prompts

Identical for both checkpoints; full text in `run_config.json` → `prompts`. PersianMedQA: Persian system prompt + options A–D + ≤3-sentence CoT + mandated final line `پاسخ: X`. PerMedCQA: Persian clinician system prompt, patient age/sex as context, ≤150-word Persian answer.

## 5. PersianMedQA results

| model | served_model | n | api_errors | accuracy | ci_low | ci_high | se | accuracy_unweighted | macro_f1 | format_failure_rate | truncated_rate | mean_latency_s | mean_completion_tokens |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| medgemma-27b-text | google/medgemma-27b-text-it | 1000 | 0 | 0.6635 | 0.6367 | 0.6903 | 0.0137 | 0.665 | 0.6655 | 0.001 | 0.003 | 10.17 | 156.1 |

Metrics: `accuracy` is the design-weighted estimate for the whole split (sum of W_h * p_h) with a 95% interval from the stratified variance including the finite-population correction; `accuracy_unweighted` is the raw sample mean, for comparison. Plus macro-F1 across the four options (catches option bias), format-failure rate (reply with no extractable choice) and truncation rate. Items whose API call failed are excluded and counted under `api_errors`.

## 6. PerMedCQA results

| model | n | api_errors | medical_correctness | relevance | completeness | persian_fluency | terminology | patient_language_understanding | judge_mean | persian_ratio | lcs_f1 | length_ratio | truncated | mean_latency_s |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| medgemma-27b-text | 500 | 0 | 3.552 | 4.473 | 3.428 | 4.905 | 4.406 | 4.394 | 4.193 | 0.992 | 0.067 | 10.42 | 0 | 16 |

Judge: `gpt-6-sol` — independent of the models under test.

Six dimensions scored 1–5 (medical correctness, relevance, completeness, Persian fluency, terminology, patient-language understanding), reference-guided against the physician's reply, one answer at a time so there is no position bias. Automatic metrics alongside: Persian-character ratio (language drift), token-overlap F1 against the expert reply, length ratio, truncation rate.

## 7. Persian understanding and medical terminology

Persian-character ratio per model: `medgemma-27b-text` 0.992; terminology score: `medgemma-27b-text` 4.41/5; patient-language understanding: `medgemma-27b-text` 4.39/5.

Per-question-type judge means (which patient intents break down) are in `summary.json`; the taxonomy tally below is the evidence for recurring errors.

## 8. Recurring errors

| error | medgemma-27b-text |
|---|---|
| missing_key_info | 270 |
| hallucinated_fact | 116 |
| contradicts_expert | 64 |
| wrong_terminology | 58 |
| misread_patient_question | 48 |
| unsafe_advice | 42 |
| generic_deferral_only | 11 |
| overlong_or_repetitive | 6 |
| non_persian_or_mixed_language | 3 |
| off_topic | 1 |

## 9. Examples

Correct / incorrect MCQ answers, best / worst open-ended answers, and a random unfiltered sample of 10 items per dataset are in `qualitative_examples.json` and rendered in section 11 of the notebook.

## 10. Cost and execution time

| Phase | Wall seconds |
|---|---|
| load_datasets | 16.5 |
| persianmedqa::medgemma-27b-text | 1275.2 |
| permedcqa::medgemma-27b-text | 1006.5 |
| permedcqa::judge | 394.6 |
| **total** | **2809.0** |

At `CONCURRENCY=8`. Client-side cost is negligible; the constraint is GPU memory on the serving side — MedGemma-27B in bf16 is ~54 GB of weights, so one 80 GB H100/A100 (or two 40 GB cards with tensor parallelism). The judge runs on OpenAI's API and costs tokens, not GPU time.

## 11. Caveats

* Single greedy sample per item — no self-consistency, no prompt-variation study.
* The judge is a single LLM. Its scores are calibrated by nothing but its own prior; a ~30-item human review on the same six dimensions is the recommended confirmation step before quoting them.
* Accuracy is computed over items that returned a reply; API errors are reported separately rather than silently scored as wrong.